In [1]:
from EC_GNN import Mobility_ECGNN
from gnn.ECC_GNN_V2 import MCC_GNN
from advancedgnn import Advanced_Mobility_GNN

In [72]:
import torch
from torch.utils.data import random_split
from torch_geometric.loader import DataLoader


# dataset = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/graphswitharea.pt', weights_only=False)
# dataset = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/graphssmaller.pt', weights_only=False)
dataset = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/morediversetesting2000.pt', weights_only=False)

# dataset = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/morediversetestinggravity2000.pt', weights_only=False)

#dataset = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/morediversetestingbigger1000.pt', weights_only=False)
total_graphs = len(dataset)

#train test split
train_size = int(0.8 * total_graphs)
test_size = total_graphs - train_size

print(f"Total Graphs: {total_graphs} | Training on: {train_size} | Testing on: {test_size}")

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

Total Graphs: 2000 | Training on: 1600 | Testing on: 400


In [73]:
import torch
import torch.nn as nn

# 1. Calculate the average data.y from the training set
total_y = 0.0
total_samples = 0

# We iterate through the train_loader to ensure we handle any batching correctly
global_min = 100
global_max = 0

for data in train_loader:
    total_y += data.y.sum().item()
    total_samples += data.y.numel()  # numel() gets the total number of elements
    batch_min = data.y.min().item()
    batch_max = data.y.max().item()

    # Update the global values if the batch values are more extreme
    if batch_min < global_min:
        global_min = batch_min

    if batch_max > global_max:
        global_max = batch_max

average_y = total_y / total_samples
print(f"Training Baseline Average (Guess): {average_y:.4f}")
print (f"max: {global_max}, min: {global_min}")

# 2. Define the L1 Loss function
criterion = nn.L1Loss()

# 3. Evaluate this average guess on the test set
total_test_loss = 0.0
test_batches = 0

for data in test_loader:
    # Create a prediction tensor filled with the average value,
    # explicitly cast to float to avoid issues if data.y is an integer type
    preds = torch.full_like(data.y, average_y, dtype=torch.float)

    # Calculate the L1 loss between our static guess and the actual targets
    loss = criterion(preds, data.y)

    total_test_loss += loss.item()
    test_batches += 1

# Calculate the mean L1 loss across all test batches
baseline_l1_loss = total_test_loss / test_batches

print(f"Baseline Test L1 Loss: {baseline_l1_loss:.4f}")

Training Baseline Average (Guess): 0.6050
max: 0.9900645613670349, min: 0.500000536441803
Baseline Test L1 Loss: 0.0642


In [12]:
import torch

def save_checkpoint(model, optimizer, epoch, loss, filepath="gnn_checkpoint.pth"):
    """Saves the full state of the training process."""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss
    }
    torch.save(checkpoint, filepath)
    print(f"Checkpoint saved at epoch {epoch} to {filepath}")

In [121]:
def load_checkpoint(model, optimizer, filepath="gnn_checkpoint.pth"):
    """Loads the saved state back into the model and optimizer."""
    # Load the dictionary from the file
    checkpoint = torch.load(filepath, map_location='cpu')

    # Inject the saved weights and momentum
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Retrieve the last epoch and loss
    start_epoch = checkpoint['epoch']
    last_loss = checkpoint['loss']

    print(f"Resumed training from epoch {start_epoch} (Previous Loss: {last_loss:.4f})")

    return model, optimizer, start_epoch

In [119]:
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader

# 1. Initialize your model
#model = Mobility_ECGNN(node_features_dim=3, edge_features_dim=1, hidden_dim=64)
# model = MCC_GNN(edge_dimension=1, node_dimension= 4, hidden_dimension= 64)
#model = Advanced_Mobility_GNN(edge_dimension=1, node_dimension= 4, hidden_dimension= 64)

model, optimizer, start_epoch = load_checkpoint(model, optimizer, "gnn_testing_radiation_wihtout_position_smoothingloss.pth")

optimizer = torch.optim.Adam(model.parameters(), lr=0.00001, weight_decay=1e-4) # Adam is the standard for GNNs
#criterion = nn.MSELoss() #MSE
# criterion = nn.L1Loss()
criterion = nn.SmoothL1Loss()

# training_graphs = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/testingpyg.pt', weights_only=False)
# train_loader = DataLoader(training_graphs, batch_size=32, shuffle=True)

#training loop
epochs = 100
FEATURE_TO_IGNORE = [2,3]

for epoch in range(epochs):
    total_loss = 0

    model.train()
    for batch_data in train_loader:

        batch_data.x[:, FEATURE_TO_IGNORE] = 0.0

        optimizer.zero_grad()
        predictions =  model(batch_data)

        loss = criterion(predictions.squeeze(), batch_data.y)
        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    model.eval()
    total_test_error = 0

    with torch.no_grad():
     for batch_data in test_loader:
        batch_data.x[:, FEATURE_TO_IGNORE] = 0.0

        # Make the prediction (remember, use model(), not model.forward()!)
        predictions = model(batch_data)

        # Calculate the Mean Squared Error (MSE) for this batch
        loss = criterion(predictions.squeeze(), batch_data.y)
        total_test_error += loss.item()

# 3. Calculate final average error
     avg_test_mse = total_test_error / len(test_loader)


    if epoch % 1 == 0:
        print(f"Epoch {epoch} | Average Training Loss (MSE): {avg_loss:.6f}")
        print(f"Epoch {epoch} | Average Testing Loss (MSE): {avg_test_mse:.6f}")
        save_checkpoint(model, optimizer, epoch, loss, "gnn_testing_radiation_wihtout_position_smoothinglosskkkk.pth")

Resumed training from epoch 4 (Previous Loss: 0.0002)
Epoch 0 | Average Training Loss (MSE): 0.000287
Epoch 0 | Average Testing Loss (MSE): 0.000252
Checkpoint saved at epoch 0 to gnn_testing_radiation_wihtout_position_smoothingloss.pth
Epoch 1 | Average Training Loss (MSE): 0.000267
Epoch 1 | Average Testing Loss (MSE): 0.000253
Checkpoint saved at epoch 1 to gnn_testing_radiation_wihtout_position_smoothingloss.pth


KeyboardInterrupt: 

In [245]:
model, optimizer, start_epoch = load_checkpoint(model, optimizer, "gnn_testing_radiation_wihtout_position_smoothingloss_gpu_fully.pth")

Resumed training from epoch 99 (Previous Loss: 0.0001)


In [1]:
testing_graphs = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/testingpyg2.pt', weights_only=False)
test_loader = DataLoader(testing_graphs, batch_size=32, shuffle=True)

NameError: name 'torch' is not defined

In [179]:
model.eval()

# We will track the total error across all test graphs
total_test_error = 0.0

# 2. Turn off Autograd
# We use torch.no_grad() because we are not updating weights.
# This makes the forward pass incredibly fast and uses almost zero memory.
with torch.no_grad():
    for batch_data in test_loader:
        batch_data.x[:, FEATURE_TO_IGNORE] = 0.0

        # Make the prediction (remember, use model(), not model.forward()!)
        predictions = model(batch_data)

        # Calculate the Mean Squared Error (MSE) for this batch
        loss = criterion(predictions.squeeze(), batch_data.y)
        total_test_error += loss.item()

# 3. Calculate final average error
avg_test_mse = total_test_error / len(test_loader)

print(f"Final Test MSE: {avg_test_mse:.6f}")

KeyboardInterrupt: 

In [253]:
# real_graph = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/georgia.pt', weights_only=False)

georgia = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/georgiacovid.pt', weights_only=False)
london = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/london.pt', weights_only=False)
japan = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/japan_data.pt', weights_only=False)

# 2. Combine them into a simple list (This is your dataset!)
complete_dataset = [georgia]

# georgia = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/georgiacovid.pt', weights_only=False)

real_loader = DataLoader(complete_dataset, batch_size=1, shuffle=True)

In [255]:
from torchmetrics import R2Score

model.eval()

total_test_error = 0.0

# Define which feature index you want to ignore (0, 1, or 2)
# For example, if your features are [Age, Income, Height], and you want
# to ignore 'Income', you would set this to 1.
FEATURE_TO_IGNORE = [2,3]
r2_metric = R2Score()

with torch.no_grad():
    for batch_data in real_loader:

        # 1. Zero out the specific feature column for all nodes in the batch
        # We use a slice [:, index] to select all rows, but only that specific column
        batch_data.x[:, FEATURE_TO_IGNORE] = 0.0

        # 2. Make the prediction (the model still sees 3 columns, but one is empty)
        predictions = model(batch_data)

        # 3. Calculate the Mean Squared Error (MSE)
        loss = criterion(predictions.view_as(batch_data.y), batch_data.y)
        total_test_error += loss.item()
        # r2_metric.update(predictions.view_as(batch_data.y), batch_data.y)

avg_test_mse = total_test_error / len(real_loader)
# final_r2 = r2_metric.compute()

print(f"Final Test (ignoring feature {FEATURE_TO_IGNORE}): {avg_test_mse:.6f}")
# print(f"final r2 {final_r2:.6f}")

r2_metric.reset()

Final Test (ignoring feature [2, 3]): 0.000613


In [256]:
batch_data.y

tensor([0.8662])

In [257]:
predictions

tensor([[0.8312]])

In [196]:
model.eval()

# We will track the total error across all test graphs
total_test_error = 0.0

# 2. Turn off Autograd
# We use torch.no_grad() because we are not updating weights.
# This makes the forward pass incredibly fast and uses almost zero memory.
with torch.no_grad():
    for batch_data in real_loader:

        # Make the prediction (remember, use model(), not model.forward()!)
        predictions = model(batch_data)

        # Calculate the Mean Squared Error (MSE) for this batch
        loss = criterion(predictions.squeeze(), batch_data.y)
        total_test_error += loss.item()

# 3. Calculate final average error
avg_test_mse = total_test_error / len(real_loader)

print(f"Final Test: {avg_test_mse:.6f}")

Final Test: 0.000001


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:1042: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.smooth_l1_loss(input, target, reduction=self.reduction, beta=self.beta)


In [41]:
predictions

tensor([[1.0922]])